In [3]:
import sys
print(sys.executable)

/opt/anaconda3/envs/interview-anon/bin/python


# 01 - VoxCeleb1 Exploration

## Objective

Explore VoxCeleb1 as a candidate dataset for speaker identity and privacy evaluation.

## Questions

- How many speakers are available?
- How many utterances are available per speaker?
- How many recordings are available per speaker?
- Are there enough independent recordings for train/validation/test splits?
- Can the English-only subset be identified reliably?

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [12]:
DATA_DIR = Path.home() / "Documents" / "voxceleb1" / "wav"

print("Dataset path:", DATA_DIR)
print("Exists:", DATA_DIR.exists())

Dataset path: /Users/juni/Documents/voxceleb1/wav
Exists: True


In [13]:
speakers = [p for p in DATA_DIR.iterdir() if p.is_dir()]

print("Number of speakers:", len(speakers))

Number of speakers: 1211


In [14]:
rows = []

for speaker_dir in speakers:
    video_dirs = [p for p in speaker_dir.iterdir() if p.is_dir()]
    wav_files = list(speaker_dir.rglob("*.wav"))

    rows.append({
        "speaker_id": speaker_dir.name,
        "n_videos": len(video_dirs),
        "n_utterances": len(wav_files)
    })

In [15]:
import pandas as pd

df = pd.DataFrame(rows)
df.head()

,speaker_id,n_videos,n_utterances
0,id10384,8,53
1,id11064,16,52
2,id10370,19,105
3,id11090,19,211
4,id10142,13,72


In [18]:
df.describe()

np.int64(21820)

In [21]:
df["n_videos"].sum()

np.int64(21820)

In [22]:
df["n_utterances"].sum()

np.int64(148642)

In [17]:
df.sort_values("n_utterances", ascending=False).head(20)

,speaker_id,n_videos,n_utterances
735,id10986,32,1002
183,id10096,16,656
974,id10020,27,589
291,id11111,23,570
647,id10710,26,489
1119,id10935,22,476
549,id11211,31,468
582,id10397,29,426
653,id10945,26,409
206,id10203,29,401


In [24]:
from sklearn.model_selection import train_test_split

In [25]:
video_rows = []

for speaker_dir in speakers:
    for video_dir in speaker_dir.iterdir():
        if video_dir.is_dir():
            wav_files = list(video_dir.glob("*.wav"))

            video_rows.append({
                "speaker_id": speaker_dir.name,
                "video_id": video_dir.name,
                "n_utterances": len(wav_files)
            })

video_df = pd.DataFrame(video_rows)
video_df.head()

,speaker_id,video_id,n_utterances
0,id10384,vxBFGKGXSFA,7
1,id10384,qXG2i6-PxZY,15
2,id10384,t-zurNhQm8A,2
3,id10384,ozGl4eZyZ_E,3
4,id10384,Ay1xwkYsNAE,1


In [26]:
videos_per_speaker = (
    video_df.groupby("speaker_id")
    .size()
    .sort_values(ascending=False)
)

videos_per_speaker.describe()

count    1211.000000
mean       18.018167
std         5.981763
min         8.000000
25%        13.000000
50%        17.000000
75%        22.000000
max        36.000000
dtype: float64

In [27]:
eligible_speakers = videos_per_speaker[videos_per_speaker >= 3]

print("Speakers with at least 3 videos:", len(eligible_speakers))

Speakers with at least 3 videos: 1211


In [28]:
import numpy as np

rng = np.random.default_rng(42)

split_rows = []

for speaker_id, group in video_df.groupby("speaker_id"):
    videos = group["video_id"].tolist()

    if len(videos) < 3:
        continue

    rng.shuffle(videos)

    n = len(videos)

    n_train = max(1, int(0.7 * n))
    n_val = max(1, int(0.15 * n))

    # Make sure at least one video remains for test
    if n_train + n_val >= n:
        n_train = n - 2
        n_val = 1

    train_videos = videos[:n_train]
    val_videos = videos[n_train:n_train + n_val]
    test_videos = videos[n_train + n_val:]

    for video_id in train_videos:
        split_rows.append({
            "speaker_id": speaker_id,
            "video_id": video_id,
            "split": "train"
        })

    for video_id in val_videos:
        split_rows.append({
            "speaker_id": speaker_id,
            "video_id": video_id,
            "split": "val"
        })

    for video_id in test_videos:
        split_rows.append({
            "speaker_id": speaker_id,
            "video_id": video_id,
            "split": "test"
        })

split_df = pd.DataFrame(split_rows)

split_df.head()

,speaker_id,video_id,split
0,id10001,1zcIwhmdeo4,train
1,id10001,7gWzIy6yIIk,train
2,id10001,J9lHsKG98U8,train
3,id10001,utrA-v8pPm4,train
4,id10001,zELwAz2W6hM,train


In [29]:
split_df["split"].value_counts()

split
train    14757
test      4386
val       2677
Name: count, dtype: int64

In [31]:
duplicates = (
    split_df.groupby(["speaker_id", "video_id"])["split"]
    .nunique()
)

print("Videos appearing in multiple splits:",
      (duplicates > 1).sum())

Videos appearing in multiple splits: 0


In [32]:
split_df.groupby("split")["speaker_id"].nunique()

split
test     1211
train    1211
val      1211
Name: speaker_id, dtype: int64

In [33]:
split_df = split_df.merge(
    video_df,
    on=["speaker_id", "video_id"],
    how="left"
)

In [34]:
split_df.groupby("split")["n_utterances"].sum()

split
test      29662
train    100962
val       18018
Name: n_utterances, dtype: int64